In [15]:
import seaborn as sns
import pandas as pd

df = sns.load_dataset('titanic')
print(f'Rindu skaits: {df.shape[0]}')
print(f'Kolonnu skaits: {df.shape[1]}')
df.head()

Rindu skaits: 891
Kolonnu skaits: 15


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [16]:
df.shape

(891, 15)

In [17]:
df.columns.tolist()

['survived',
 'pclass',
 'sex',
 'age',
 'sibsp',
 'parch',
 'fare',
 'embarked',
 'class',
 'who',
 'adult_male',
 'deck',
 'embark_town',
 'alive',
 'alone']

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [19]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


### A. Problēmas apraksts

Mūsu galvenais mērķis, izmantojot šo Titānika datu kopu, ir izveidot modeli, kas spētu prognozēt, vai konkrēts pasažieris izdzīvos vai nē kā arī no kura klāja vai kajītes viņš ir. Tas kā Titānika datu analītiķim palīdzētu saprast, no kuriem stāviem vai kajītēm pasažieri visvairāk izdzīvoja un visvairāk nomira, lai tālāk varētu analizēt iemeslus, piemēram, kuģa uzbūvi, pasažieru izvietojumu un katastrofas cēloni.Gala lietotājs ir kuģa būvnieks un ceļojumu plānotājs.


### B. ML uzdevuma tips

Šis ir **supervised learning** (uzraudzītās mācīšanās) uzdevums, jo mūsu vēsturiskajos datos jau ir iedota "pareizā atbilde". Mums ir kolonna `survived`, kas skaidri norāda, vai konkrētais pasažieris izdzīvoja vai nē. Modelis mācīsies no šiem gatavajiem piemēriem.

Konkrētāk, šis ir **klasifikācijas** uzdevums. Mēs mēģinām datoram iemācīt iedalīt cilvēkus divās konkrētās kategorijās (izdzīvoja - 1, neizdzīvoja - 0), nevis prognozēt kādu nepārtrauktu skaitli (kā tas būtu regresijas gadījumā).

In [20]:
# Target mainīgais
print(df['survived'].value_counts())
print(f'\nIzdzīvoja: {df["survived"].mean():.1%}')

survived
0    549
1    342
Name: count, dtype: int64

Izdzīvoja: 38.4%


### C. Target variable

Mūsu mērķa mainīgais (target variable) ir kolonna `survived`. Tā ir galvenā vērtība, kuru mūsu modelis mēģinās iemācīties prognozēt. 
Šajā kolonnā ir tikai divas iespējamās vērtības:
* **0** — pasažieris neizdzīvoja.
* **1** — pasažieris izdzīvoja.

### D. Features saraksts

Lai modelis varētu prognozēt izdzīvošanu, tam ir nepieciešami dati (features), no kuriem mācīties. Mūsu datu kopā šīs iezīmes var sadalīt divās lielās grupās:

1. [cite_start]**Skaitliskās (numerical) features:** Tās ir kolonnas, kas satur reālus skaitļus un mērvienības, piemēram: `age` (vecums), `fare` (biļetes cena), `sibsp` (brāļu/māsu/laulāto skaits), `parch` (vecāku/bērnu skaits)[cite: 145].
2. [cite_start]**Kategoriskās (categorical) features:** Tās ir kolonnas, kas sadala cilvēkus grupās, piemēram: `sex` (dzimums), `pclass` (biļetes klase), `embarked` (iekāpšanas osta), `alone` (vai ceļoja viens)[cite: 146].

In [21]:
# Features pārskats
print('Skaitliskās features:')
print(df.select_dtypes(include='number').columns.tolist())

print('\nKategoriskās features:')
print(df.select_dtypes(include='category').columns.tolist())

Skaitliskās features:
['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']

Kategoriskās features:
['class', 'deck']


### E. Sākotnējie novērojumi

Ātri aplūkojot datus, var pamanīt vairākas interesantas lietas. Pirmkārt, kopējais izdzīvojušo skaits ir salīdzinoši neliels – izdzīvoja tikai aptuveni 38% no visiem pasažieriem. Otrkārt, datos ir ievērojams trūkstošo vērtību daudzums, īpaši kolonnās `age` (vecums) un `deck` (klājs), kas nozīmē, ka pirms modeļa trenēšanas šie dati būs īpaši jāsagatavo un jāapstrādā.